# 20 — XGB paired export  (auto-generado por build_notebook_20_xgb_paired.py)

Exporta las predicciones **por muestra** de B5_XGB sobre el split de **test**
para los tres corredores (**E2**, **E59**, **E4**) a horizontes
**h ∈ {1, 3, 5, 10}**, con la clave completa
`(corridor, direction, horizon, t, pair_rank)`.

**Por qué**: las métricas agregadas de XGB
(`baselines_results_multih.csv`) se calculan sobre TODAS las filas de test con
predicción no nula, mientras que las métricas DL se calculan sobre la población
de ventanas (sin cold-start, con el target replicado por slot de anclaje). El
sesgo de encuadre medido para la persistencia (0.28-0.53 min) es MAYOR que 7 de
los 8 márgenes reclamados frente a XGB, así que la comparación agregada no es
defendible. Para re-puntuar XGB sobre exactamente las mismas filas que ve el DL
hace falta la clave única de la tabla de headways, y `pair_rank` es
imprescindible: `t` sola NO es única (~4.2 filas por `(t, direction)`).

**Este kernel NO reescribe ningún artefacto congelado.** `headways_E4.parquet`
se **monta** desde `16-e4-data-baselines` y nunca se regenera: su SHA-256 está
congelado en los INPUT_HASHES de NB17/NB18/NB19. Este notebook no escribe
parquet alguno.

## Setup — gate de hashes congelados y rutas de salida

Los CUATRO inputs pasan por el gate: los tres parquets de headways y
`atypical_days.csv`. Los digests son los MISMOS que congelan los notebooks DL
(NB11/NB13 para E2+E59, NB17/NB19 para E4), porque este export sólo es
comparable con los residuos DL si XGB se ajustó sobre bytes idénticos.

In [ ]:

import hashlib
import time

import polars as pl
import numpy as np
from pathlib import Path

# Frozen SHA-256 of every input. Identical digests to the DL notebooks:
#   headways_E2/E59  → NB11/NB12/NB13 INPUT_HASHES
#   headways_E4      → NB17/NB18/NB19 INPUT_HASHES
#   atypical_days    → all of the above + NB10/NB16
# The parquets are hash-gated here (NB10 did not gate them) because a paired
# re-scoring is only valid if XGB saw byte-identical inputs to the DL models.
INPUT_HASHES = {
    "headways_E2.parquet": "82a34eaffc79cd82346d4595a2e72f5d3ffb751ed37fa0fc0cde3a8f8fb345d4",
    "headways_E59.parquet": "0b5f5593caaa94e4e6af7da672bc2cad7b49b69b7cbd0a22092f15700a89a448",
    "headways_E4.parquet": "1dde7f38eea9bc7d9941c17cbc3d326cb864e70be815a1a7e3d0ae2691f19273",
    "atypical_days.csv": "2054245cc830e58b9397b75ea3b55d034581046b64e73b1630ca7d464e3ecb86",
}

def _sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

# Locate a required input by filename and verify its frozen SHA-256.
def _resolve_input(name: str) -> Path:
    roots = [Path("/kaggle/input"), Path(".")]
    candidates = [p for root in roots if root.exists() for p in sorted(root.rglob(name))]
    if not candidates:
        raise FileNotFoundError(f"Required input not found anywhere: {name}")
    for path in candidates:
        if _sha256_file(path) == INPUT_HASHES[name]:
            return path
    raise ValueError(
        f"No copy of {name} matches its frozen SHA-256 — "
        f"candidates: {[str(p) for p in candidates]}"
    )

OUTPUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
OUTPUT_DIR.mkdir(exist_ok=True)

# Output names avoid `_results_` (globbed by build_degradation_curve.py),
# `_residuals_h` (globbed by evaluation/paired_audit.py) and `_multiseed_`
# (globbed by evaluation/multiseed.py) so no analysis-layer discovery can pick
# them up and crash on the foreign schema.
PAIRED_OUT = OUTPUT_DIR / "xgb_paired_persample_test.csv"
PROVENANCE_OUT = OUTPUT_DIR / "xgb_paired_search_provenance.csv"

HORIZONS = [1, 3, 5, 10]
CORRIDORS = [("E2", 2), ("E59", 59), ("E4", 4)]

print(f"Output dir: {OUTPUT_DIR}")
print(f"Paired export: {PAIRED_OUT}")
print(f"Search provenance: {PROVENANCE_OUT}")

## Module: evaluation/splits

Split temporal (`split_temporal`) y winsorización train-only p99
(`winsorize_train_p99`). El umbral p99 se calcula SOLO en train y se aplica a
TODOS los splits — `run_corridor` le pasa el frame completo con la etiqueta de
split, así que el contrato viaja intacto hasta este export.

In [ ]:
"""Temporal split and winsorization helpers for headway evaluation — Fase 3.

Public API:
    split_temporal(df: pl.DataFrame) -> pl.DataFrame
    winsorize_train_p99(df: pl.DataFrame) -> tuple[pl.DataFrame, float]

Constants (split date ranges, locked in spec §3 and design §5):
    SPLIT_TRAIN_START, SPLIT_TRAIN_END
    SPLIT_VAL_START,   SPLIT_VAL_END
    SPLIT_TEST_START,  SPLIT_TEST_END
    WINSOR_QUANTILE

Design decisions (locked in design §5 and §9):
  - Split key is pl.col("t").dt.date() membership, NOT row index.
  - Three ranges are exhaustive and mutually exclusive.
  - Rows outside all three ranges receive None (split column = null).
  - Winsorization threshold is computed on train rows only (AC-WINSOR-1, AC-WINSOR-2).
  - Null delta_t_min rows are NOT clipped (AC-WINSOR-3).
  - Rows above threshold are clipped (not dropped) (AC-WINSOR-4).
  - Constants live here (not PRODUCTIVE_PARAMS) — evaluation protocol concern.
  - WINSOR_QUANTILE and split dates are not added to pyproject.toml.
"""
from __future__ import annotations

from datetime import date

import polars as pl

# ---------------------------------------------------------------------------
# Split date range constants (spec §3, inclusive on both ends)
# ---------------------------------------------------------------------------

SPLIT_TRAIN_START: date = date(2023, 10, 1)
SPLIT_TRAIN_END:   date = date(2024, 1, 15)

SPLIT_VAL_START:   date = date(2024, 1, 16)
SPLIT_VAL_END:     date = date(2024, 2, 7)

SPLIT_TEST_START:  date = date(2024, 2, 8)
SPLIT_TEST_END:    date = date(2024, 2, 29)

WINSOR_QUANTILE: float = 0.99


def split_temporal(df: pl.DataFrame) -> pl.DataFrame:
    """Add a `split` column (Utf8) with values {"train", "val", "test"}.

    Membership is determined by pl.col("t").dt.date() against the six
    module-level date constants.  Rows outside all three ranges receive
    null (should not exist in the R7 v4 dataset; harness raises if found).

    Parameters
    ----------
    df:
        headways DataFrame containing at least a `t` (Datetime) column.

    Returns
    -------
    pl.DataFrame — input frame with one added column `split: Utf8`.
    """
    day = pl.col("t").dt.date()
    return df.with_columns(
        pl.when((day >= SPLIT_TRAIN_START) & (day <= SPLIT_TRAIN_END))
          .then(pl.lit("train"))
          .when((day >= SPLIT_VAL_START) & (day <= SPLIT_VAL_END))
          .then(pl.lit("val"))
          .when((day >= SPLIT_TEST_START) & (day <= SPLIT_TEST_END))
          .then(pl.lit("test"))
          .otherwise(None)
          .alias("split")
    )


def winsorize_train_p99(
    df: pl.DataFrame,
) -> tuple[pl.DataFrame, float]:
    """Clip delta_t_min to the 99th-percentile threshold computed on train rows only.

    The threshold is computed once as a scalar from non-null train-split rows.
    It is then applied as a clip ceiling to ALL rows (train + val + test).
    Null delta_t_min values are never clipped — they remain null (AC-WINSOR-3).

    Parameters
    ----------
    df:
        headways DataFrame that already has a `split` column (added by
        split_temporal) and a `delta_t_min` (Float64 nullable) column.

    Returns
    -------
    (clipped_df, threshold)
        clipped_df: same schema as df, delta_t_min clipped.
        threshold: the scalar train-p99 value used as the clip ceiling.

    Design note (AC-WINSOR-2 leakage guard):
        The filter `split == "train"` is applied BEFORE computing the quantile,
        so extreme outliers in val or test rows cannot shift the threshold.
    """
    threshold = float(
        df.filter(
            (pl.col("split") == "train") & pl.col("delta_t_min").is_not_null()
        )["delta_t_min"]
        .quantile(WINSOR_QUANTILE)
    )

    # Clip: preserve null rows; clip non-null rows to threshold from above.
    # pl.min_horizontal(col, lit(threshold)) would coerce null → 0 in some
    # polars versions, so we use the explicit when/then pattern (design §5).
    clipped = df.with_columns(
        pl.when(pl.col("delta_t_min").is_null())
          .then(None)
          .otherwise(
              pl.min_horizontal(pl.col("delta_t_min"), pl.lit(threshold))
          )
          .alias("delta_t_min")
    )
    return clipped, threshold

## Module: evaluation/metrics

`mae` y `rmse` en minutos (los consume `harness`; este notebook exporta residuos
por muestra, no métricas agregadas).

In [ ]:
"""Evaluation metrics for headway forecasting — Fase 3.

Public API:
    mae(y_true, y_pred) -> float
    rmse(y_true, y_pred) -> float

Both functions accept polars Series (Float64) or numpy arrays (float64).
Null / NaN masking: rows where EITHER y_true or y_pred is null/NaN are
dropped before aggregation.  If no valid rows remain, ValueError is raised.

Design decisions locked in design §4:
  - ValueError on empty/all-null input (NOT silent NaN return).
  - Only MAE and RMSE are in scope (spec B3-NO-MAPE — ratio-based metrics
    are out of scope because near-zero headways cause denominator blow-up).
  - No new pyproject.toml dependencies (polars + numpy already present).
"""
from __future__ import annotations

import numpy as np
import polars as pl


def _to_numpy_with_mask(
    y_true: pl.Series | np.ndarray,
    y_pred: pl.Series | np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    """Coerce both inputs to float64 numpy arrays and apply the null/NaN mask.

    Polars Series with dtype Float64: null cells become NaN via .to_numpy().
    numpy arrays: assumed to already use NaN for missing values.

    Returns
    -------
    (y_true_masked, y_pred_masked) — two 1-D float64 arrays of equal length,
    containing no NaN values.  May be empty if all rows were masked.
    """
    # Coerce to numpy.
    if isinstance(y_true, pl.Series):
        yt = y_true.to_numpy(allow_copy=True).astype(np.float64)
    else:
        yt = np.asarray(y_true, dtype=np.float64).ravel()

    if isinstance(y_pred, pl.Series):
        yp = y_pred.to_numpy(allow_copy=True).astype(np.float64)
    else:
        yp = np.asarray(y_pred, dtype=np.float64).ravel()

    # Elementwise mask: keep row only if BOTH sides are finite (not NaN).
    mask = ~(np.isnan(yt) | np.isnan(yp))
    return yt[mask], yp[mask]


def mae(
    y_true: pl.Series | np.ndarray,
    y_pred: pl.Series | np.ndarray,
) -> float:
    """Mean Absolute Error in minutes, with null/NaN masking.

    Parameters
    ----------
    y_true, y_pred:
        Ground-truth and predicted headway values in minutes.
        Accepts polars Series (Float64) or numpy arrays (float64).
        Null / NaN positions in either input are dropped before computation.

    Returns
    -------
    float — MAE in minutes.

    Raises
    ------
    ValueError
        If the masked input is empty (all-null or zero-length).
    """
    yt, yp = _to_numpy_with_mask(y_true, y_pred)
    if len(yt) == 0:
        raise ValueError(
            "mae: metric on empty/all-null input — no valid (y_true, y_pred) pairs."
        )
    return float(np.mean(np.abs(yt - yp)))


def rmse(
    y_true: pl.Series | np.ndarray,
    y_pred: pl.Series | np.ndarray,
) -> float:
    """Root Mean Squared Error in minutes, with null/NaN masking.

    Parameters
    ----------
    y_true, y_pred:
        Ground-truth and predicted headway values in minutes.
        Accepts polars Series (Float64) or numpy arrays (float64).
        Null / NaN positions in either input are dropped before computation.

    Returns
    -------
    float — RMSE in minutes.

    Raises
    ------
    ValueError
        If the masked input is empty (all-null or zero-length).
    """
    yt, yp = _to_numpy_with_mask(y_true, y_pred)
    if len(yt) == 0:
        raise ValueError(
            "rmse: metric on empty/all-null input — no valid (y_true, y_pred) pairs."
        )
    return float(np.sqrt(np.mean((yt - yp) ** 2)))

## Module: baselines/statistical

B0-B4. Aquí importa sobre todo **B1** (persistencia naive, horizon-aware): es la
columna `y_pred_persist` del export emparejado.

In [ ]:
"""Classical statistical baselines for headway forecasting — Fase 3.

Public API:
    predict_b0(headways: pl.DataFrame) -> pl.DataFrame
    predict_b1(headways: pl.DataFrame, *, horizon: int = 1) -> pl.DataFrame
    predict_b2(headways: pl.DataFrame, *, window: int, horizon: int = 1) -> pl.DataFrame
    predict_b3(headways: pl.DataFrame, *, alpha: float = SES_ALPHA, horizon: int = 1) -> pl.DataFrame
    predict_b4_ha(headways: pl.DataFrame) -> pl.DataFrame

Input contract (all four functions):
    The DataFrame must have a `split` column (Utf8) added by split_temporal.
    Columns consumed: empresaid, t, direction, pair_rank, delta_t_min, split.

Output contract:
    Each function returns the input frame with ONE additional column:
        B0 → y_pred_b0
        B1 → y_pred_b1
        B2 → y_pred_b2_w{window}
        B3 → y_pred_b3

Predictions are filled for ALL rows (train + test); the evaluation harness
consumes test rows only.  Filling train rows costs negligibly more and lets
future SDDs reuse predictions if needed (design §3).

Design decisions locked in design §3 and §9:
  - Functions, not classes (consistent with project precedent).
  - Slot key: (empresaid, direction, pair_rank).
  - B2 `window` = count of last NON-NULL observations (not a time window).
  - B2 min_periods = window // 2  (floor division).
  - B3 alpha = SES_ALPHA = 0.3, per-slot online recursion, null-skip.
  - B3 state init: NaN until first non-null train obs; first non-null sets s directly.
  - No new pyproject.toml dependencies (polars + numpy only).
"""
from __future__ import annotations

import numpy as np
import polars as pl

# ---------------------------------------------------------------------------
# Module-level constants (locked in design §9)
# ---------------------------------------------------------------------------

BASELINE_B2_WINDOWS: tuple[int, ...] = (5, 10, 15)
SES_ALPHA: float = 0.3

_SLOT_COLS: list[str] = ["empresaid", "direction", "pair_rank"]


# ===========================================================================
# B0 — Global mean per slot (train rows only)
# ===========================================================================

def predict_b0(headways: pl.DataFrame) -> pl.DataFrame:
    """Add column `y_pred_b0`: per-slot mean of train delta_t_min.

    The prediction is constant within a slot — the arithmetic mean of all
    non-null delta_t_min values in the train split for that slot.  Slots
    with no non-null train observations receive null (AC-B0-2).

    Parameters
    ----------
    headways:
        headways DataFrame with `split` column attached.

    Returns
    -------
    pl.DataFrame — input frame with `y_pred_b0` (Float64 nullable) added.
    """
    train_means = (
        headways
        .filter(pl.col("split") == "train")
        .group_by(_SLOT_COLS)
        .agg(pl.col("delta_t_min").mean().alias("y_pred_b0"))
    )
    return headways.join(train_means, on=_SLOT_COLS, how="left")


# ===========================================================================
# B1 — Naive / persistence baseline
# ===========================================================================

def predict_b1(headways: pl.DataFrame, *, horizon: int = 1) -> pl.DataFrame:
    """Add column `y_pred_b1`: last non-null delta_t_min seen `horizon` steps before each row.

    Uses forward_fill().shift(horizon).over(slot) — the canonical polars pattern for
    ŷ_{t+h} = y_t with null gaps.  Causal by construction (shift prevents the
    current-row value from appearing as its own prediction).

    Parameters
    ----------
    headways:
        headways DataFrame with `split` column attached.
    horizon:
        Number of steps to shift. Default 1 reproduces the original behavior.
        Calling ``predict_b1(df)`` (no horizon arg) is identical to ``predict_b1(df, horizon=1)``.

    Returns
    -------
    pl.DataFrame — input frame sorted by (slot, t), with `y_pred_b1` added.
    """
    return (
        headways
        .sort(_SLOT_COLS + ["t"])
        .with_columns(
            pl.col("delta_t_min")
              .forward_fill()
              .shift(horizon)
              .over(_SLOT_COLS)
              .alias("y_pred_b1")
        )
    )


# ===========================================================================
# B2 — Trailing moving average of last w NON-NULL observations
# ===========================================================================

def predict_b2(headways: pl.DataFrame, *, window: int, horizon: int = 1) -> pl.DataFrame:
    """Add column `y_pred_b2_w{window}`: mean of last `window` non-null observations.

    Semantics (locked, design §3):
      - `window` is a COUNT of non-null observations, not a time window.
      - min_periods = window // 2  (floor).
      - Prediction at row i uses only observations strictly before row i (causal).

    Horizon rule (Fase 6.5):
      The 1-step prediction is computed first (rolling_mean.shift(1) on the
      non-null sub-series + join_asof backward).  Then shift(horizon-1) is
      applied over the slot key so that ŷ_{t+h} = ŷ_{t+1} lagged by h-1
      additional steps.  horizon=1 → shift(0) = identity (backward-compatible).

    Implementation:
      - group_by(slot).map_groups(lambda g: _b2_one_slot(g, window))
      - Within each group: extract non-null values, compute rolling_mean with
        shift(1) (causal), then join_asof(strategy="backward") back to
        original group rows on `t`.
      - After concat, apply shift(horizon-1).over(_SLOT_COLS).

    Parameters
    ----------
    headways:
        headways DataFrame with `split` column attached.
    window:
        Number of non-null observations in the trailing window.
    horizon:
        Prediction horizon in steps.  Default 1 reproduces the original
        behavior exactly.

    Returns
    -------
    pl.DataFrame — input frame with `y_pred_b2_w{window}` (Float64 nullable) added.
    """
    col_name = f"y_pred_b2_w{window}"
    min_periods = window // 2

    def _b2_one_slot(group: pl.DataFrame) -> pl.DataFrame:
        group = group.sort("t")

        # Extract non-null rows only, in time order.
        non_null = group.filter(pl.col("delta_t_min").is_not_null())

        if len(non_null) == 0:
            # No non-null observations: all predictions are null.
            return group.with_columns(pl.lit(None, dtype=pl.Float64).alias(col_name))

        # Compute rolling mean on the non-null sub-series, then shift(1) for
        # causality: the prediction at position i uses observations 0..i-1.
        non_null = non_null.with_columns(
            pl.col("delta_t_min")
              .rolling_mean(window_size=window, min_samples=min_periods)
              .shift(1)
              .alias(col_name)
        )

        # Align back to the full group (including null rows) via join_asof.
        # strategy="backward" finds the most recent non-null rolling mean at or
        # before each timestamp in the original group.
        result = group.join_asof(
            non_null.select(["t", col_name]),
            on="t",
            strategy="backward",
        )
        return result

    sorted_df = headways.sort(_SLOT_COLS + ["t"])
    slots = sorted_df.partition_by(_SLOT_COLS, maintain_order=True)
    result = pl.concat([_b2_one_slot(g) for g in slots])

    # Apply horizon shift: shift(horizon-1) over slot so predictions look
    # h steps ahead.  shift(0) is a no-op → backward-compatible for horizon=1.
    if horizon > 1:
        result = (
            result
            .sort(_SLOT_COLS + ["t"])
            .with_columns(
                pl.col(col_name)
                  .shift(horizon - 1)
                  .over(_SLOT_COLS)
                  .alias(col_name)
            )
        )

    return result


# ===========================================================================
# B3 — Simple Exponential Smoothing (α=0.3, per slot, online)
# ===========================================================================

def _ses_one_slot(slot_df: pl.DataFrame, alpha: float) -> pl.DataFrame:
    """Online SES recursion for a single slot.

    s_t = α·y_t + (1-α)·s_{t-1}  (null observations skip the update).
    pred[i] = s before observing y[i]  (causal: shift-1 semantics).

    Initialization: s = NaN until the first non-null y_t; the first non-null
    value sets s directly (no prior needed) — AC-B3-3.  The prediction at
    that initialization row is NaN (no prior state), so the first test
    prediction for a slot with at least one train observation is the state
    after consuming ALL train rows.
    """
    slot_df = slot_df.sort("t")
    y = slot_df["delta_t_min"].to_numpy(allow_copy=True).astype(np.float64)
    pred = np.full(len(y), np.nan)
    s = np.nan  # smoothing state; NaN until first non-null

    for i in range(len(y)):
        # Prediction at row i is the state BEFORE observing y[i].
        pred[i] = s
        # Update state if current observation is not null/NaN.
        if not np.isnan(y[i]):
            if np.isnan(s):
                s = y[i]  # initialization: first non-null sets state directly
            else:
                s = alpha * y[i] + (1.0 - alpha) * s

    # Convert float NaN → polars null so downstream is_null() works correctly.
    pred_series = pl.Series("y_pred_b3", pred, dtype=pl.Float64)
    return slot_df.with_columns(pred_series.set(pred_series.is_nan(), None))


def predict_b3(headways: pl.DataFrame, *, alpha: float = SES_ALPHA, horizon: int = 1) -> pl.DataFrame:
    """Add column `y_pred_b3`: online SES predictions, α=0.3 (default).

    Applies the causal recursion s_t = α·y_t + (1-α)·s_{t-1} per slot.
    Null observations do not update the state (AC-B3-2).
    State is initialized from the first non-null observation (AC-B3-3).
    Slots with all-null values emit null for all rows (AC-B3-4).

    Horizon rule (Fase 6.5):
      The 1-step SES predictions are computed first (existing per-slot loop).
      Then shift(horizon-1) is applied over the slot key so that ŷ_{t+h} =
      ŷ_{t+1} lagged by h-1 additional steps.
      horizon=1 → shift(0) = identity (backward-compatible).

    Parameters
    ----------
    headways:
        headways DataFrame with `split` column attached.
    alpha:
        Smoothing parameter.  Default SES_ALPHA = 0.3 (locked, design §3).
        Tests may pass alternative values for edge-case verification.
    horizon:
        Prediction horizon in steps.  Default 1 reproduces the original
        behavior exactly.

    Returns
    -------
    pl.DataFrame — input frame with `y_pred_b3` (Float64 nullable) added.

    Design note (D-PL-OVER-VS-MAPGROUPS):
        polars .over() does not support stateful numpy loops; map_groups
        materializes one Python frame per slot (~30–100 per corridor — trivially fast).
    """
    sorted_df = headways.sort(_SLOT_COLS + ["t"])
    slots = sorted_df.partition_by(_SLOT_COLS, maintain_order=True)
    result = pl.concat([_ses_one_slot(g, alpha) for g in slots])

    # Apply horizon shift: shift(horizon-1) over slot so predictions look
    # h steps ahead.  shift(0) is a no-op → backward-compatible for horizon=1.
    if horizon > 1:
        result = (
            result
            .sort(_SLOT_COLS + ["t"])
            .with_columns(
                pl.col("y_pred_b3")
                  .shift(horizon - 1)
                  .over(_SLOT_COLS)
                  .alias("y_pred_b3")
            )
        )

    return result


# ===========================================================================
# B4 — Historical Average per (slot, hour-of-day) from train only
# ===========================================================================

def predict_b4_ha(headways: pl.DataFrame) -> pl.DataFrame:
    """Add column `y_pred_b4_ha`: per-slot, per-hour mean of train delta_t_min.

    For each (empresaid, direction, pair_rank, hour) group, computes the mean
    of non-null delta_t_min values from train rows only.  Test/val rows at the
    same hour receive that mean as their prediction.  Hours not seen in train
    produce null predictions.
    """
    ha_key = _SLOT_COLS + ["_hour"]
    df = headways.with_columns(pl.col("t").dt.hour().alias("_hour"))

    train_means = (
        df
        .filter(pl.col("split") == "train")
        .group_by(ha_key)
        .agg(pl.col("delta_t_min").mean().alias("y_pred_b4_ha"))
    )

    result = df.join(train_means, on=ha_key, how="left")
    return result.drop("_hour")

## Module: data/context_features

`load_atypical_days` + `encode_context` — los MISMOS helpers que usan los
notebooks DL. B5_XGB recibe `atypical_flag` a través de ellos. Debe embeberse
ANTES de `fitted`, que importa `encode_context`.

In [ ]:
"""Context features module for supervised dataset construction — Fase 3 DL.

AC-CTX-1: encode_context adds hour_sin, hour_cos at midnight → (0, 1).
AC-CTX-2: encode_context adds dow_sin, dow_cos with period 7; emits 5 named columns.
AC-CTX-3: load_atypical_days(None) returns empty set (graceful fallback, DL-2).
AC-CTX-4: load_atypical_days(path) returns set[date] from CSV when file exists.
AC-CTX-5: atypical_flag=1.0 when timestamp date in atypical_dates, else 0.0.
AC-CTX-6: zero torch imports at module level (INV-10, DL-10).

Design decisions locked in design §2.4 and §5:
  - encode_context operates on a DataFrame with a `t` (Datetime) column.
  - Cyclical encoding: sin(2π * value / period), cos(2π * value / period).
  - atypical_flag = 1.0 when t.date() in atypical_dates else 0.0.
  - DL-2: graceful fallback to atypical_flag=0 when path is None or missing.
  - No torch imports (INV-10).
"""
from __future__ import annotations

import logging
import math
import warnings
from datetime import date
from pathlib import Path

import polars as pl

_log = logging.getLogger(__name__)

# ---------------------------------------------------------------------------
# Constants
# ---------------------------------------------------------------------------

CONTEXT_FEATURE_NAMES: tuple[str, ...] = (
    "hour_sin",
    "hour_cos",
    "dow_sin",
    "dow_cos",
    "atypical_flag",
)


# ---------------------------------------------------------------------------
# Internal helpers
# ---------------------------------------------------------------------------

def _cyclical_pair(col: pl.Expr, period: int, prefix: str) -> list[pl.Expr]:
    """Emit [sin_expr, cos_expr] aliased <prefix>_sin, <prefix>_cos.

    Encoding: sin(2π * col / period), cos(2π * col / period).

    Parameters
    ----------
    col:
        Polars expression that yields a numeric value (e.g. hour 0-23, dow 0-6).
    period:
        Full cycle length (24 for hour, 7 for day-of-week).
    prefix:
        Column name prefix ("hour" or "dow").
    """
    angle = col * (2.0 * math.pi / period)
    return [
        angle.sin().alias(f"{prefix}_sin"),
        angle.cos().alias(f"{prefix}_cos"),
    ]


# ---------------------------------------------------------------------------
# Public API
# ---------------------------------------------------------------------------

def encode_context(
    df: pl.DataFrame,
    *,
    atypical_dates: set[date] | None = None,
) -> pl.DataFrame:
    """Add 5 context columns derived from the `t` (Datetime) column.

    AC-CTX-1..5. DL-2 graceful fallback: atypical_flag=0 when atypical_dates
    is None or empty.

    Parameters
    ----------
    df:
        DataFrame with a `t` (Datetime[us]) column.
    atypical_dates:
        Set of dates that are atypical (e.g. holidays, strikes). When None
        or empty, atypical_flag is 0.0 for all rows.

    Returns
    -------
    pl.DataFrame — input frame with 5 additional columns appended in the order
    defined by CONTEXT_FEATURE_NAMES.
    """
    if atypical_dates is None:
        atypical_dates = set()

    # Cyclical hour and day-of-week encodings.
    # polars dt.weekday() returns ISO weekday: Monday=1 .. Sunday=7.
    # We convert to 0-indexed (Monday=0 .. Sunday=6) to align with Python convention
    # so that midnight Monday → dow=0 → dow_sin=sin(0)=0, dow_cos=cos(0)=1 (AC-CTX-1).
    hour_expr = pl.col("t").dt.hour().cast(pl.Float64)
    dow_expr = (pl.col("t").dt.weekday() - 1).cast(pl.Float64)

    sin_cos_exprs: list[pl.Expr] = [
        *_cyclical_pair(hour_expr, 24, "hour"),
        *_cyclical_pair(dow_expr, 7, "dow"),
    ]

    # Atypical flag: 1.0 if the date is in the atypical set, else 0.0.
    if atypical_dates:
        # Build a list of date literals to check membership against.
        atypical_list = sorted(atypical_dates)
        date_col = pl.col("t").dt.date()
        flag_expr = pl.lit(0.0)

        # Chain when/then for each atypical date.
        flag_chain = pl.when(
            date_col == pl.lit(atypical_list[0])
        ).then(pl.lit(1.0))
        for d in atypical_list[1:]:
            flag_chain = flag_chain.when(
                date_col == pl.lit(d)
            ).then(pl.lit(1.0))
        flag_expr = flag_chain.otherwise(pl.lit(0.0))
    else:
        flag_expr = pl.lit(0.0)

    return df.with_columns(
        *sin_cos_exprs,
        flag_expr.cast(pl.Float64).alias("atypical_flag"),
    )


def load_atypical_days(
    path: Path | str | None,
) -> set[date]:
    """Read CSV with at least a `date` column; return set[date].

    AC-CTX-3 + DL-2: returns empty set when path is None OR file does not exist.
    A warning is emitted when the path is non-None but missing (so callers know
    the fallback was triggered — not a silent failure).

    Parameters
    ----------
    path:
        Path to a CSV file with a `date` column (ISO-8601 format).
        May be None, a string, or a Path object.

    Returns
    -------
    set[date] — parsed dates, or empty set on fallback.
    """
    if path is None:
        return set()

    resolved = Path(path)
    if not resolved.exists():
        warnings.warn(
            f"load_atypical_days: file not found at '{resolved}'; "
            "falling back to empty atypical set (atypical_flag=0 for all rows). "
            "DL-2 graceful fallback.",
            stacklevel=2,
        )
        return set()

    df = pl.read_csv(resolved, try_parse_dates=True)
    # The frozen 02-eda-corridors CSV names its date column `day`; older
    # fixtures use `date`. Accept either, preferring `date` when both exist.
    date_col = next((c for c in ("date", "day") if c in df.columns), None)
    if date_col is None:
        warnings.warn(
            f"load_atypical_days: CSV at '{resolved}' has no 'date' or 'day' column; "
            "falling back to empty set.",
            stacklevel=2,
        )
        return set()

    dates: set[date] = set()
    for val in df[date_col].to_list():
        if val is not None:
            if isinstance(val, date):
                dates.add(val)
            else:
                try:
                    from datetime import datetime as _dt
                    dates.add(_dt.fromisoformat(str(val)).date())
                except ValueError:
                    _log.warning("Skipping unparseable date value: %s", val)

    return dates

## Module: baselines/fitted

`fit_predict_b5_xgb` — baseline ajustado B5_XGB (12 lags + calendario/slot +
flag de día atípico) con random search sembrado de 24 configuraciones elegidas
**solo** sobre validación. `B5FitResult.predictions` CONSERVA `pair_rank`: es la
puerta de entrada del export emparejado.

In [ ]:
"""Fitted ML baseline for headway forecasting — gradient-boosted regressor (B5_XGB).

Why this module is separate from `statistical.py`:
    B0-B4 are closed-form/recursive predictors with NO learned parameters and a
    "no new dependencies" design lock. B5_XGB is a *fitted* learner (XGBoost) —
    a different category. It answers the reviewer reflex "where is a fitted/ML
    baseline?" that pure naive baselines (persistence, moving average, SES,
    historical average) do not.

Design — fair comparison to the DL models (NB11-13, NB17-19):
    The DL models consume an input window of T_in = 12 consecutive 1-minute
    steps and predict the headway HORIZON steps after the last input step. The
    XGBoost baseline is given the SAME information: 12 lagged headway values
    ending HORIZON steps before the target, so `lag_1` equals the B1 persistence
    prediction (`shift(horizon)`) and the model strictly extends the naive
    baselines rather than seeing extra future data. Calendar context (hour,
    weekday), static slot keys (direction, pair_rank) and the atypical-day flag
    round out the features.

    Two asymmetries versus the DL models were removed (peer-review fix):
      1. ATYPICAL-DAY FLAG. The DL models receive `atypical_flag` as a required,
         hash-pinned context feature. B5_XGB now receives the same binary flag,
         built with the SAME `encode_context` helper so the semantics cannot
         drift between the two model families.
      2. HYPERPARAMETER SEARCH. The DL models were tuned; B5_XGB used a single
         hardcoded configuration. It now runs a seeded random search of
         `SEARCH_N_CONFIGS` configurations selected STRICTLY on the validation
         split (see `_random_search`), with the winning configuration reported
         back to the caller so it can be persisted and audited.

Contract (mirrors statistical.py):
    predict_b5_xgb(headways, *, horizon=1, seed=42, atypical_dates=None,
                   search=True) -> headways + y_pred_b5_xgb
    fit_predict_b5_xgb(...) -> B5FitResult (predictions + search provenance)

    Input must have the `split` column (added by split_temporal). The model is
    fit on TRAIN rows only; predictions are produced for ALL rows. Validation
    rows are used for hyperparameter selection and early stopping ONLY when
    there are enough of them (>= _MIN_VAL_ROWS); otherwise the frozen default
    configuration and a fixed number of trees are used.

Leakage contract (hard):
    The `test` split NEVER influences training, early stopping, or
    hyperparameter selection. Selection reads the validation loss only.

Atypical-flag contract (mirrors the DL notebooks):
    `atypical_dates=None` means "no atypical calendar supplied" (library/fixture
    use) and yields an all-zero flag column. Passing an EXPLICIT EMPTY SET is a
    configuration error — a CSV that parsed to nothing must fail closed instead
    of silently disabling the feature — and raises ValueError.

Determinism:
    Fixed `seed`, fixed `SEARCH_SEED` for the configuration sampler, and
    `tree_method="hist"` with a pinned `nthread` → repeated calls on the same
    machine with the same inputs produce identical predictions and select the
    same configuration.
"""
from __future__ import annotations

from dataclasses import dataclass, field
from datetime import date

import numpy as np
import polars as pl


_SLOT_COLS: list[str] = ["empresaid", "direction", "pair_rank"]

# Number of lagged headway steps fed to the model = DL input window (T_in).
N_LAGS: int = 12

# Use validation rows for search + early stopping only when there are at least
# this many; tiny test fixtures (and corridors with no val rows) fall back to
# the frozen default configuration and a fixed number of trees.
_MIN_VAL_ROWS: int = 50

# Threads for the Kaggle CPU kernel (4 vCPU). Pinned in source: XGBoost `hist`
# is reproducible for a FIXED thread count, so this value is part of the
# determinism contract and must not be made environment-dependent.
_NTHREAD: int = 4

_NUM_BOOST_ROUND: int = 400
_EARLY_STOPPING_ROUNDS: int = 30

# Cheaper budget for the search sweep; the winner is refit at the full budget.
_SEARCH_NUM_BOOST_ROUND: int = 200
_SEARCH_EARLY_STOPPING_ROUNDS: int = 20

# Frozen fallback configuration (the pre-search hardcoded baseline). Used when
# there is no usable validation split, or when `search=False`.
_XGB_PARAMS: dict = {
    "eta": 0.05,
    "max_depth": 6,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "min_child_weight": 5,
    "lambda": 1.0,
    "objective": "reg:squarederror",
    "tree_method": "hist",
    "nthread": _NTHREAD,
}

# ---------------------------------------------------------------------------
# Hyperparameter random search — validation-only selection.
# ---------------------------------------------------------------------------

# EXACTLY 24 configurations: the agreed budget for a Kaggle CPU kernel that
# must fit 2 corridors x 4 horizons within the session runtime limit.
SEARCH_N_CONFIGS: int = 24

# Fixed in source so the search is reproducible and cannot be silently
# re-rolled between runs. Changing this value changes the paper's numbers.
SEARCH_SEED: int = 20240718

# Discrete search space (|space| = 6*6*5*5*5*5 = 22500 >> 24 draws).
SEARCH_SPACE: dict[str, list] = {
    "eta": [0.02, 0.03, 0.05, 0.08, 0.12, 0.20],
    "max_depth": [3, 4, 5, 6, 8, 10],
    "min_child_weight": [1, 3, 5, 10, 20],
    "subsample": [0.6, 0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.6, 0.7, 0.8, 0.9, 1.0],
    "lambda": [0.5, 1.0, 2.0, 5.0, 10.0],
}


def sample_search_configs(
    n_configs: int = SEARCH_N_CONFIGS, *, seed: int = SEARCH_SEED
) -> list[dict]:
    """Draw `n_configs` DISTINCT hyperparameter configurations, deterministically.

    Sampling depends only on `seed` and `SEARCH_SPACE` — never on the data — so
    every corridor and horizon evaluates the same candidate set and the sweep is
    exactly reproducible.

    Returns
    -------
    list[dict] — each dict holds only the searched keys (eta, max_depth,
    min_child_weight, subsample, colsample_bytree, lambda).
    """
    rng = np.random.default_rng(seed)
    keys = sorted(SEARCH_SPACE)  # sorted → draw order independent of dict order
    seen: set[tuple] = set()
    configs: list[dict] = []
    # Bounded loop: the space is ~22500 wide, so 24 distinct draws are reached
    # almost immediately; the cap only guards against a shrunken space.
    for _ in range(n_configs * 1000):
        if len(configs) == n_configs:
            break
        values = tuple(
            SEARCH_SPACE[k][int(rng.integers(len(SEARCH_SPACE[k])))] for k in keys
        )
        if values in seen:
            continue
        seen.add(values)
        configs.append(dict(zip(keys, values)))
    if len(configs) != n_configs:
        raise ValueError(
            f"sample_search_configs: could only draw {len(configs)} distinct "
            f"configurations out of {n_configs} requested"
        )
    return configs


@dataclass(frozen=True)
class B5FitResult:
    """Predictions plus the provenance needed to audit the fitted baseline.

    Attributes
    ----------
    predictions:
        Input frame (sorted by slot, t) with `y_pred_b5_xgb` added.
    best_params:
        The full parameter dict handed to XGBoost for the final fit.
    best_val_rmse:
        Validation RMSE of the selected configuration (``nan`` when no search
        ran, i.e. no usable validation split).
    best_iteration:
        Boosting iteration chosen by early stopping (-1 when unavailable).
    n_configs_evaluated:
        How many configurations the search actually fit (0 when it was skipped).
    search_seed:
        Seed used to draw the candidate configurations.
    used_atypical_flag:
        True when a non-empty atypical calendar was supplied.
    """

    predictions: pl.DataFrame
    best_params: dict = field(default_factory=dict)
    best_val_rmse: float = float("nan")
    best_iteration: int = -1
    n_configs_evaluated: int = 0
    search_seed: int = SEARCH_SEED
    used_atypical_flag: bool = False


def _build_features(
    headways: pl.DataFrame,
    *,
    horizon: int,
    atypical_dates: set[date] | None = None,
) -> tuple[pl.DataFrame, list[str]]:
    """Return the frame sorted by (slot, t) with lag + context feature columns
    added, plus the list of feature column names.

    lag_k (k = 1..N_LAGS) = headway value (forward-filled within slot) observed
    `horizon + k - 1` steps before the target row. lag_1 == B1 persistence.

    `_atypical` is the SAME binary flag the DL models consume: it is produced by
    `encode_context`, not reimplemented here, so the two model families cannot
    diverge on what counts as an atypical day.

    Raises
    ------
    ValueError
        If `atypical_dates` is an explicit empty set (fail closed — see the
        module docstring's atypical-flag contract).
    """
    if atypical_dates is not None and len(atypical_dates) == 0:
        raise ValueError(
            "_build_features: atypical_dates parsed to an EMPTY set. The "
            "atypical-day feature must not be silently disabled — pass None "
            "only when no atypical calendar exists at all."
        )

    lag_exprs = [
        pl.col("delta_t_min")
        .forward_fill()
        .shift(horizon + k - 1)
        .over(_SLOT_COLS)
        .alias(f"_lag_{k}")
        for k in range(1, N_LAGS + 1)
    ]
    df = (
        encode_context(headways, atypical_dates=atypical_dates)
        .sort(_SLOT_COLS + ["t"])
        .with_columns(
            *lag_exprs,
            pl.col("t").dt.hour().alias("_hour"),
            pl.col("t").dt.weekday().alias("_weekday"),
            pl.col("atypical_flag").alias("_atypical"),
        )
    )
    feature_cols = (
        [f"_lag_{k}" for k in range(1, N_LAGS + 1)]
        + ["_hour", "_weekday", "direction", "pair_rank", "_atypical"]
    )
    return df, feature_cols


def _random_search(
    xgb,
    dtrain,
    dval,
    *,
    seed: int,
    n_configs: int,
    search_seed: int,
) -> tuple[dict, float, int, int]:
    """Fit `n_configs` candidates and keep the one with the lowest VALIDATION RMSE.

    The only signal read here is `booster.best_score` on `dval` — the test split
    is not part of either DMatrix, so selection cannot see it.

    Returns
    -------
    (best_params, best_val_rmse, best_iteration, n_evaluated)
    """
    best_params: dict = {}
    best_score = float("inf")
    best_iteration = -1
    n_evaluated = 0

    for candidate in sample_search_configs(n_configs, seed=search_seed):
        params = dict(_XGB_PARAMS, **candidate, seed=seed)
        booster = xgb.train(
            params,
            dtrain,
            num_boost_round=_SEARCH_NUM_BOOST_ROUND,
            evals=[(dval, "val")],
            early_stopping_rounds=_SEARCH_EARLY_STOPPING_ROUNDS,
            verbose_eval=False,
        )
        n_evaluated += 1
        score = float(booster.best_score)
        # Strict `<` → first-drawn config wins ties, keeping selection
        # deterministic for a fixed candidate order.
        if score < best_score:
            best_score = score
            best_params = params
            best_iteration = int(getattr(booster, "best_iteration", -1))

    return best_params, best_score, best_iteration, n_evaluated


def fit_predict_b5_xgb(
    headways: pl.DataFrame,
    *,
    horizon: int = 1,
    seed: int = 42,
    atypical_dates: set[date] | None = None,
    search: bool = True,
    n_configs: int = SEARCH_N_CONFIGS,
    search_seed: int = SEARCH_SEED,
) -> B5FitResult:
    """Fit B5_XGB and return predictions plus the auditable search provenance.

    Parameters
    ----------
    headways:
        headways DataFrame with the `split` column attached. Columns consumed:
        empresaid, t, direction, pair_rank, delta_t_min, split.
    horizon:
        Forecast horizon in steps. lag_1 = shift(horizon) so the 1-lag feature
        equals B1 persistence; horizon=1 is the default.
    seed:
        XGBoost training seed (reproducible tree construction).
    atypical_dates:
        Atypical-day calendar (the same set the DL models receive). None means
        "not supplied" → all-zero flag; an explicit empty set raises.
    search:
        When True (default) run the seeded validation-only random search. When
        False, or when there is no usable validation split, the frozen default
        configuration is used.
    n_configs / search_seed:
        Search budget and sampler seed. Both default to the frozen constants;
        overriding them is a test/debug affordance, not a production path.

    Returns
    -------
    B5FitResult
    """
    import xgboost as xgb

    original_cols = headways.columns
    df, feature_cols = _build_features(
        headways, horizon=horizon, atypical_dates=atypical_dates
    )
    used_atypical = bool(atypical_dates)

    is_train = df["split"] == "train"
    is_val = df["split"] == "val"
    target_present = df["delta_t_min"].is_not_null()

    train_mask = (is_train & target_present).to_numpy()
    n_train = int(train_mask.sum())

    # Degenerate: nothing to fit on → null predictions (mirrors B0 on empty slots).
    if n_train == 0:
        return B5FitResult(
            predictions=df.select(original_cols).with_columns(
                pl.lit(None, dtype=pl.Float64).alias("y_pred_b5_xgb")
            ),
            used_atypical_flag=used_atypical,
        )

    X_all = df.select(feature_cols).to_numpy().astype(np.float64)
    y_all = df["delta_t_min"].to_numpy().astype(np.float64)

    dtrain = xgb.DMatrix(X_all[train_mask], label=y_all[train_mask], missing=np.nan)
    dall = xgb.DMatrix(X_all, missing=np.nan)

    val_mask = (is_val & target_present).to_numpy()
    has_val = int(val_mask.sum()) >= _MIN_VAL_ROWS

    params = dict(_XGB_PARAMS, seed=seed)
    best_val_rmse = float("nan")
    best_iteration = -1
    n_evaluated = 0

    if has_val:
        # NOTE: dval holds VALIDATION rows only. The test split is absent from
        # every DMatrix built here, so neither early stopping nor hyperparameter
        # selection can read it.
        dval = xgb.DMatrix(X_all[val_mask], label=y_all[val_mask], missing=np.nan)
        if search:
            params, best_val_rmse, _search_iter, n_evaluated = _random_search(
                xgb,
                dtrain,
                dval,
                seed=seed,
                n_configs=n_configs,
                search_seed=search_seed,
            )
        # Refit the selected configuration at the full boosting budget, keeping
        # the existing early-stopping-on-validation behaviour.
        booster = xgb.train(
            params,
            dtrain,
            num_boost_round=_NUM_BOOST_ROUND,
            evals=[(dval, "val")],
            early_stopping_rounds=_EARLY_STOPPING_ROUNDS,
            verbose_eval=False,
        )
        best_val_rmse = float(booster.best_score)
        best_iteration = int(getattr(booster, "best_iteration", -1))
    else:
        booster = xgb.train(params, dtrain, num_boost_round=_NUM_BOOST_ROUND)

    preds = booster.predict(dall).astype(np.float64)

    return B5FitResult(
        predictions=df.select(original_cols).with_columns(
            pl.Series("y_pred_b5_xgb", preds, dtype=pl.Float64)
        ),
        best_params=dict(params),
        best_val_rmse=best_val_rmse,
        best_iteration=best_iteration,
        n_configs_evaluated=n_evaluated,
        search_seed=search_seed,
        used_atypical_flag=used_atypical,
    )


def predict_b5_xgb(
    headways: pl.DataFrame,
    *,
    horizon: int = 1,
    seed: int = 42,
    atypical_dates: set[date] | None = None,
    search: bool = True,
) -> pl.DataFrame:
    """Add column `y_pred_b5_xgb`: gradient-boosted forecast of delta_t_min.

    Thin wrapper over :func:`fit_predict_b5_xgb` for callers that only need the
    predictions. See that function for the full parameter documentation.

    Returns
    -------
    pl.DataFrame — input frame (sorted by slot, t) with `y_pred_b5_xgb`
        (Float64 nullable) added. If the train split has no usable rows, the
        column is all-null.
    """
    return fit_predict_b5_xgb(
        headways,
        horizon=horizon,
        seed=seed,
        atypical_dates=atypical_dates,
        search=search,
    ).predictions

## Module: baselines/harness

`run_corridor` compone split → winsorize → B0-B4 + B5_XGB → métricas. Se embebe
**verbatim y sin modificar**: es el mismo código inlineado en NB10/NB16, y
tocarlo rompería la identidad byte a byte de esos notebooks congelados.

In [ ]:
"""Evaluation harness for classical baseline comparison — Fase 3.

Public API:
    run_corridor(headways, corridor_name, ...) -> CorridorRun
        metrics + per-sample XGB residuals + fitted-baseline provenance.
    evaluate_corridor(headways: pl.DataFrame, corridor_name: str) -> pl.DataFrame
        Metrics only (thin wrapper over run_corridor; unchanged contract).

The function composes the full pipeline for one corridor:
    split_temporal → winsorize_train_p99 → predict_b0/b1/b2(×3)/b3/b4_ha
    [→ predict_b5_xgb when include_fitted]
    → filter test rows → compute MAE + RMSE per (direction, baseline)
    → return tidy long-form DataFrame.

Output schema (design §6):
    corridor   Utf8
    direction  Utf8   — "-1", "+1", "aggregate"
    baseline   Utf8   — "B0", "B1", "B2_w5", "B2_w10", "B2_w15", "B3", "B4_HA"
                        [, "B5_XGB" when include_fitted]
    metric     Utf8   — "MAE", "RMSE"
    value      Float64 — minutes

Rows per corridor: 3 directions × N baselines × 2 metrics.
    include_fitted=True  (default): N = 8 → 48 rows per corridor.
    include_fitted=False (formulaic-only): N = 7 → 42 rows per corridor.

Design decisions (locked in design §6 and §9):
  - "aggregate" direction = MAE/RMSE over POOLED test rows (both directions
    concatenated), NOT mean of per-direction metrics.
  - val rows are NEVER consumed by the formulaic baselines B0-B4 (B3-VAL-UNUSED).
    The fitted baseline B5_XGB MAY use val rows for early stopping (only when
    there are enough), which is correct practice for a learned model and mirrors
    how the DL models were tuned.
  - B5_XGB (the fitted ML baseline) adds an xgboost dependency; it lives in
    fitted.py and is opt-out via include_fitted=False.
  - harness.py does NOT read parquets or write CSV (notebook does those).
"""
from __future__ import annotations

from dataclasses import dataclass, field
from datetime import date

import polars as pl


# Map from prediction column name → display name for the output DataFrame.
_BASELINE_MAP: list[tuple[str, str]] = [
    ("y_pred_b0", "B0"),
    ("y_pred_b1", "B1"),
    ("y_pred_b2_w5", "B2_w5"),
    ("y_pred_b2_w10", "B2_w10"),
    ("y_pred_b2_w15", "B2_w15"),
    ("y_pred_b3", "B3"),
    ("y_pred_b4_ha", "B4_HA"),
]

# The fitted ML baseline is appended only when include_fitted=True.
_FITTED_ENTRY: tuple[str, str] = ("y_pred_b5_xgb", "B5_XGB")

# Per-sample residual export for the paired significance tests (DM / Wilcoxon).
# `t` is the join key: without it the XGBoost residuals cannot be paired with
# any other model's per-sample errors.
XGB_RESIDUAL_COLUMNS: list[str] = [
    "corridor",
    "direction",
    "horizon",
    "t",
    "y_true",
    "y_pred_xgb",
    "y_pred_persist",
]


@dataclass(frozen=True)
class CorridorRun:
    """Everything one corridor x horizon run produces.

    Attributes
    ----------
    metrics:
        Tidy long-form MAE/RMSE table (the historical `evaluate_corridor` output).
    residuals:
        Per-sample TEST residuals for B5_XGB paired with B1 persistence, with the
        `t` join key. Empty frame when `include_fitted=False`.
    fit_result:
        Provenance of the fitted baseline (winning hyperparameters, validation
        RMSE, search budget). None when `include_fitted=False`.
    """

    metrics: pl.DataFrame
    residuals: pl.DataFrame = field(default_factory=lambda: pl.DataFrame([]))
    fit_result: B5FitResult | None = None


def _direction_label(direction_val: int) -> str:
    """Signed direction label ("-1" / "+1") shared with the DL residual exports."""
    return f"+{direction_val}" if direction_val > 0 else str(direction_val)


def _build_xgb_residuals(
    test_df: pl.DataFrame, corridor_name: str, horizon: int
) -> pl.DataFrame:
    """Per-sample paired TEST residuals: B5_XGB vs B1 persistence.

    Keeps only samples where the target AND both predictions are present — the
    paired set the significance tests require.
    """
    return (
        test_df.filter(
            pl.col("delta_t_min").is_not_null()
            & pl.col("y_pred_b5_xgb").is_not_null()
            & pl.col("y_pred_b1").is_not_null()
        )
        .with_columns(
            pl.lit(corridor_name).alias("corridor"),
            pl.col("direction")
            .map_elements(_direction_label, return_dtype=pl.Utf8)
            .alias("direction"),
            pl.lit(horizon, dtype=pl.Int64).alias("horizon"),
            pl.col("delta_t_min").cast(pl.Float64).alias("y_true"),
            pl.col("y_pred_b5_xgb").cast(pl.Float64).alias("y_pred_xgb"),
            pl.col("y_pred_b1").cast(pl.Float64).alias("y_pred_persist"),
        )
        .select(XGB_RESIDUAL_COLUMNS)
        .sort(["corridor", "direction", "t"])
    )


def run_corridor(
    headways: pl.DataFrame,
    corridor_name: str,
    *,
    horizon: int = 1,
    include_fitted: bool = True,
    atypical_dates: set[date] | None = None,
) -> CorridorRun:
    """Full pipeline for one corridor: metrics + XGB residuals + fit provenance.

    Same pipeline and contracts as :func:`evaluate_corridor` (which delegates
    here and returns only `metrics`), plus the two artifacts the paper needs to
    defend the fitted baseline: the per-sample paired residuals and the winning
    hyperparameter configuration.

    Parameters
    ----------
    atypical_dates:
        Atypical-day calendar forwarded to B5_XGB so the fitted baseline sees
        the same context feature as the DL models. An explicit empty set raises
        (fail closed); None means no calendar was supplied.
    """
    # --- Pipeline: split → winsorize → all baselines ---
    # INV-6: the p99 threshold is computed on TRAIN only and applied to ALL
    # splits — winsorize_train_p99 receives the full split-tagged frame.
    df = split_temporal(headways)
    df, _threshold = winsorize_train_p99(df)

    df = predict_b0(df)
    df = predict_b1(df, horizon=horizon)
    for w in BASELINE_B2_WINDOWS:
        df = predict_b2(df, window=w, horizon=horizon)
    df = predict_b3(df, horizon=horizon)
    df = predict_b4_ha(df)

    baseline_map = list(_BASELINE_MAP)
    fit_result: B5FitResult | None = None
    if include_fitted:
        fit_result = fit_predict_b5_xgb(
            df, horizon=horizon, atypical_dates=atypical_dates
        )
        df = fit_result.predictions
        baseline_map = baseline_map + [_FITTED_ENTRY]

    # --- Filter to test rows only (B3-VAL-UNUSED) ---
    test_df = df.filter(pl.col("split") == "test")

    metrics = _metrics_table(test_df, corridor_name, baseline_map)
    residuals = (
        _build_xgb_residuals(test_df, corridor_name, horizon)
        if include_fitted
        else pl.DataFrame([])
    )
    return CorridorRun(metrics=metrics, residuals=residuals, fit_result=fit_result)


def _metrics_table(
    test_df: pl.DataFrame,
    corridor_name: str,
    baseline_map: list[tuple[str, str]],
) -> pl.DataFrame:
    """MAE/RMSE per (direction x baseline) over the TEST rows, long-form."""
    rows: list[dict] = []

    for pred_col, baseline_name in baseline_map:
        for direction_val in (-1, 1, "aggregate"):
            if direction_val == "aggregate":
                # Pool all test rows regardless of direction.
                subset = test_df
                direction_str = "aggregate"
            else:
                subset = test_df.filter(pl.col("direction") == direction_val)
                direction_str = _direction_label(direction_val)

            y_true = subset["delta_t_min"]
            y_pred = subset[pred_col]

            # Compute MAE and RMSE (null rows are masked inside the functions).
            mae_val = mae(y_true, y_pred)
            rmse_val = rmse(y_true, y_pred)

            rows.append(
                {
                    "corridor": corridor_name,
                    "direction": direction_str,
                    "baseline": baseline_name,
                    "metric": "MAE",
                    "value": mae_val,
                }
            )
            rows.append(
                {
                    "corridor": corridor_name,
                    "direction": direction_str,
                    "baseline": baseline_name,
                    "metric": "RMSE",
                    "value": rmse_val,
                }
            )

    return pl.DataFrame(rows).with_columns(
        pl.col("corridor").cast(pl.Utf8),
        pl.col("direction").cast(pl.Utf8),
        pl.col("baseline").cast(pl.Utf8),
        pl.col("metric").cast(pl.Utf8),
        pl.col("value").cast(pl.Float64),
    )


def evaluate_corridor(
    headways: pl.DataFrame,
    corridor_name: str,
    *,
    horizon: int = 1,
    include_fitted: bool = True,
    atypical_dates: set[date] | None = None,
) -> pl.DataFrame:
    """Run all classical baselines on one corridor and return a tidy metrics table.

    Thin wrapper over :func:`run_corridor` kept for the existing callers that
    only need the metrics table.

    Parameters
    ----------
    headways:
        Raw headways DataFrame with R7 v4 schema columns:
        empresaid, t, direction, pair_rank, delta_t_min.
        Must NOT already have a `split` column (this function adds it).
    corridor_name:
        Label for the `corridor` column in the output (e.g. "E2", "E59").
    horizon:
        Prediction horizon in steps. Default 1 reproduces the original behavior
        exactly. B1, B2, B3, and B5_XGB are horizon-aware and receive this value.
        B0 and B4_HA are horizon-agnostic (constant/lookup predictors) and are
        not affected.
    include_fitted:
        When True (default), also runs the fitted ML baseline B5_XGB (xgboost).
        When False, only the formulaic baselines B0-B4 run (no xgboost import).

    Returns
    -------
    pl.DataFrame — tidy long-form table (48 rows with include_fitted, else 42):
        [corridor, direction, baseline, metric, value]

    Notes
    -----
    - val rows are ignored at prediction time (baselines consume train only)
      and are never included in metric computation (metrics use test rows only).
    - The "aggregate" direction row pools test rows from both directions before
      computing MAE/RMSE — it is NOT the mean of the two per-direction metrics.
    """
    return run_corridor(
        headways,
        corridor_name,
        horizon=horizon,
        include_fitted=include_fitted,
        atypical_dates=atypical_dates,
    ).metrics

## Module: baselines/paired_export

`export_paired_xgb` / `paired_xgb_test_frame` — reencuadran las filas de test de
`B5FitResult.predictions` al schema
`[corridor, empresaid, direction, horizon, t, pair_rank, y_true, y_pred_xgb,
y_pred_persist]`, con la MISMA semántica de filtrado que
`harness._build_xgb_residuals` (target y ambas predicciones no nulas) y la MISMA
convención de etiqueta de dirección (`"-1"` / `"+1"`) que los exports de
residuos DL. Debe embeberse DESPUÉS de `harness`, que es de quien importa.

In [ ]:
"""Per-sample B5_XGB TEST predictions keyed compatibly with the DL population.

Why this module exists
----------------------
The paper's claim "LSTM beats the leveled XGBoost baseline in 8/8 cells" compares
MAEs computed over DIFFERENT sample populations:

  * ``baselines_results_multih.csv`` (and its E4 twin) aggregate over EVERY test
    row that has a non-null prediction.
  * The DL metrics aggregate over the DL WINDOW population: cold-start rows are
    dropped and every target is replicated once per anchoring window slot.

The project's own measured aggregate-vs-paired framing bias for persistence is
0.28-0.53 min, which is larger than 7 of the 8 claimed XGB margins (+0.05 to
+0.41), so the claim cannot be defended from aggregate metrics. Re-scoring XGB
over exactly the DL's rows requires per-sample XGB predictions carrying the
FULL unique key of the headways frame.

``harness.XGB_RESIDUAL_COLUMNS`` cannot serve that purpose: it exports
``[corridor, direction, horizon, t, y_true, y_pred_xgb, y_pred_persist]`` and
drops ``pair_rank`` at the final ``select``. ``t`` is NOT a unique key — the
headways frame is keyed on ``(t, direction, pair_rank)`` and carries roughly 4.2
rows per ``(t, direction)`` — so those residuals cannot be joined row-for-row
against anything.

Why this module is additive instead of a fix in place
----------------------------------------------------
``harness.py``, ``fitted.py`` and ``statistical.py`` are inlined VERBATIM into
the NB10 and NB16 notebooks by ``build_notebook_*.embed_module``. Editing any of
them changes those generated notebooks' bytes and breaks their byte-identity
guards, which would force a re-push of frozen Kaggle artifacts. So nothing here
touches them. The escape hatch is ``B5FitResult.predictions``, which is the
input frame plus ``y_pred_b5_xgb`` and therefore still carries ``pair_rank``,
``empresaid`` and ``split``.

Contracts inherited unchanged (nothing is reimplemented here)
------------------------------------------------------------
Everything methodological is delegated to ``harness.run_corridor``:

  * temporal split (``split_temporal``),
  * winsorization — the p99 threshold of ``delta_t_min`` is computed on TRAIN
    only and applied to ALL splits, because ``run_corridor`` hands the full
    split-tagged frame to ``winsorize_train_p99`` before any baseline runs,
  * the B5_XGB feature construction, the seeded 24-configuration random search
    selected strictly on VALIDATION, and the leakage contract (the test split is
    absent from every DMatrix).

This module only RESHAPES the resulting test rows. It applies the exact same
filtering semantics as ``harness._build_xgb_residuals`` (keep a sample only when
the target AND both predictions are present) and the exact same ``"-1"`` / ``"+1"``
direction-label convention the DL residual exports use, so the string join keys
are compatible on both sides.

Reproducibility note
--------------------
An export run REFITS B5_XGB. That is safe by design: ``fitted.py`` pins the
search seed, the training seed and ``nthread``, so a rerun on the same inputs
selects the same configuration and produces the same predictions.
``search_provenance_row`` exports the winning configuration precisely so the
refit can be diffed against the frozen ``xgb_search_config_multih.csv`` /
``xgb_search_config_E4_multih.csv``; a mismatch there invalidates the export.

Output-shape decision: ONE combined long CSV
--------------------------------------------
The notebook that drives this module writes a SINGLE combined CSV for all three
corridors and all four horizons, with ``corridor`` and ``horizon`` as columns,
because:

  1. the consumer is one join against the DL per-sample residuals, so a single
     frame avoids a discover-and-concat step that could silently miss a file;
  2. ``corridor`` and ``horizon`` are already part of the join key, so splitting
     by them would encode key material in filenames instead of columns;
  3. every split-by-file scheme risks colliding with an existing glob. The
     analysis layer discovers artifacts by pattern — ``degradation.load_results``
     globs ``*_results_*.csv`` and ``paired_audit`` globs ``*_residuals_h*.csv``
     — and a foreign-schema file caught by either glob would crash the
     degradation build or silently contaminate ``consolidated_multihorizon.csv``
     and Figure 1. One file with a name matching NEITHER pattern is the smallest
     surface for that failure mode.
"""
from __future__ import annotations

from datetime import date

import polars as pl


# Full export schema. The four columns beyond `XGB_RESIDUAL_COLUMNS` are the
# point of this module: `pair_rank` completes the headways unique key and
# `empresaid` makes the composite data key `(empresaid, unidadid)`-compatible
# corridor identity explicit instead of implied by the `corridor` label.
XGB_PAIRED_COLUMNS: list[str] = [
    "corridor",
    "empresaid",
    "direction",
    "horizon",
    "t",
    "pair_rank",
    "y_true",
    "y_pred_xgb",
    "y_pred_persist",
]

# The unique key of one exported sample, and the deterministic sort order.
# `(t, direction, pair_rank)` is the unique key of the headways frame; the
# `corridor` and `horizon` prefix scopes it across the 12 (corridor, horizon)
# runs that land in one combined export.
XGB_PAIRED_KEY: list[str] = [
    "corridor",
    "direction",
    "horizon",
    "t",
    "pair_rank",
]

# Columns `paired_xgb_test_frame` needs from a `B5FitResult.predictions` frame.
_REQUIRED_SOURCE_COLUMNS: list[str] = [
    "empresaid",
    "t",
    "direction",
    "pair_rank",
    "delta_t_min",
    "split",
    "y_pred_b1",
    "y_pred_b5_xgb",
]

# Signed direction label, vectorised. Semantically identical to
# `harness._direction_label` ("+1" / "-1"), which the DL residual exports also
# use; `tests/baselines/test_paired_export.py` pins that equivalence so the two
# conventions cannot drift apart.
_DIRECTION_LABEL = (
    pl.when(pl.col("direction") > 0)
    .then(pl.concat_str([pl.lit("+"), pl.col("direction").cast(pl.Utf8)]))
    .otherwise(pl.col("direction").cast(pl.Utf8))
)


def paired_xgb_test_frame(
    predictions: pl.DataFrame, corridor_name: str, *, horizon: int
) -> pl.DataFrame:
    """Reshape a ``B5FitResult.predictions`` frame into the paired TEST export.

    Parameters
    ----------
    predictions:
        The frame returned as ``B5FitResult.predictions`` — the split-tagged,
        winsorized headways frame with the baseline prediction columns added.
        Must still carry ``pair_rank`` (it does: ``fit_predict_b5_xgb`` selects
        the caller's original columns before appending its prediction).
    corridor_name:
        Corridor label written to the ``corridor`` column (e.g. "E2", "E59", "E4").
    horizon:
        Forecast horizon in steps, written to the ``horizon`` column.

    Returns
    -------
    pl.DataFrame with exactly :data:`XGB_PAIRED_COLUMNS`, restricted to TEST rows
    where the target AND both predictions are non-null, sorted by
    :data:`XGB_PAIRED_KEY`.

    Raises
    ------
    ValueError
        If any column in :data:`_REQUIRED_SOURCE_COLUMNS` is absent — in
        particular if ``pair_rank`` was already dropped upstream, which would
        silently produce a non-unique key.
    """
    missing = [c for c in _REQUIRED_SOURCE_COLUMNS if c not in predictions.columns]
    if missing:
        raise ValueError(
            "paired_xgb_test_frame: predictions frame is missing required "
            f"columns {missing}. Pass B5FitResult.predictions from a "
            "run_corridor(..., include_fitted=True) call, not the residual export."
        )

    return (
        predictions.filter(
            (pl.col("split") == "test")
            # Same paired-sample semantics as harness._build_xgb_residuals.
            & pl.col("delta_t_min").is_not_null()
            & pl.col("y_pred_b5_xgb").is_not_null()
            & pl.col("y_pred_b1").is_not_null()
        )
        .with_columns(
            pl.lit(corridor_name, dtype=pl.Utf8).alias("corridor"),
            pl.col("empresaid").cast(pl.Int64),
            _DIRECTION_LABEL.alias("direction"),
            pl.lit(horizon, dtype=pl.Int64).alias("horizon"),
            pl.col("pair_rank").cast(pl.Int32),
            pl.col("delta_t_min").cast(pl.Float64).alias("y_true"),
            pl.col("y_pred_b5_xgb").cast(pl.Float64).alias("y_pred_xgb"),
            pl.col("y_pred_b1").cast(pl.Float64).alias("y_pred_persist"),
        )
        .select(XGB_PAIRED_COLUMNS)
        .sort(XGB_PAIRED_KEY)
    )


def paired_xgb_from_run(
    run: CorridorRun, corridor_name: str, *, horizon: int
) -> pl.DataFrame:
    """Paired TEST export for an already-computed :class:`CorridorRun`.

    Use this when the caller already ran ``run_corridor`` and wants both the
    metrics and the paired export without refitting.

    Raises
    ------
    ValueError
        If the run was produced with ``include_fitted=False`` (no fitted model,
        hence nothing to export).
    """
    if run.fit_result is None:
        raise ValueError(
            "paired_xgb_from_run: run has no fit_result — call run_corridor with "
            "include_fitted=True (the default) to fit B5_XGB."
        )
    return paired_xgb_test_frame(
        run.fit_result.predictions, corridor_name, horizon=horizon
    )


def export_paired_xgb(
    headways: pl.DataFrame,
    corridor_name: str,
    *,
    horizon: int = 1,
    atypical_dates: set[date] | None = None,
) -> tuple[pl.DataFrame, CorridorRun]:
    """Fit B5_XGB for one corridor x horizon and return the paired TEST export.

    Thin composition over ``harness.run_corridor`` — the split, the train-only
    p99 winsorization applied to all splits, the feature construction and the
    validation-only random search all happen there, unchanged.

    Parameters
    ----------
    headways:
        Raw headways frame (no ``split`` column), with ``empresaid`` present.
    corridor_name:
        Corridor label for the export.
    horizon:
        Forecast horizon in steps.
    atypical_dates:
        Atypical-day calendar forwarded to B5_XGB. An explicit empty set raises
        inside ``fitted._build_features`` (fail closed).

    Returns
    -------
    (paired_export, run)
        ``paired_export`` has :data:`XGB_PAIRED_COLUMNS`; ``run`` is returned so
        the caller can also persist metrics and the search provenance without
        paying for a second fit.
    """
    run = run_corridor(
        headways,
        corridor_name,
        horizon=horizon,
        include_fitted=True,
        atypical_dates=atypical_dates,
    )
    return paired_xgb_from_run(run, corridor_name, horizon=horizon), run


def search_provenance_row(
    run: CorridorRun, corridor_name: str, *, horizon: int
) -> dict:
    """Flat audit row describing the fitted configuration behind one export.

    Mirrors the row NB10/NB16 persist to ``xgb_search_config*.csv`` so the refit
    performed by the export notebook can be diffed against the frozen search
    provenance of the original runs.
    """
    if run.fit_result is None:
        raise ValueError(
            "search_provenance_row: run has no fit_result — call run_corridor "
            "with include_fitted=True (the default)."
        )
    fit = run.fit_result
    return {
        "corridor": corridor_name,
        "horizon": horizon,
        "n_configs_evaluated": fit.n_configs_evaluated,
        "search_seed": fit.search_seed,
        "best_val_rmse": fit.best_val_rmse,
        "best_iteration": fit.best_iteration,
        "used_atypical_flag": fit.used_atypical_flag,
        **{f"param_{k}": str(v) for k, v in sorted(fit.best_params.items())},
    }

## Días atípicos — input requerido y verificado por hash

`atypical_days.csv` (salida de NB02, kernel_source `alexhuaracha/02-eda-corridors`)
alimenta el flag `atypical_flag` que recibe B5_XGB. Es un input **requerido**: si
falta, si sus bytes no coinciden con el snapshot congelado, o si el set parsea
vacío, el kernel falla ANTES de ajustar nada — nunca se degrada en silencio a un
flag todo-ceros.

In [ ]:

atypical_path = _resolve_input("atypical_days.csv")
atypical_dates = load_atypical_days(atypical_path)
if not atypical_dates:
    raise ValueError(f"atypical_days.csv parsed to an empty date set: {atypical_path}")
print(f"Atypical days loaded: {len(atypical_dates)} dates (path={atypical_path})")

## Cargar headways — E2, E59 y E4 (sólo lectura)

Los tres parquets se resuelven por el gate de hashes. `empresaid` es implícito en
el nombre del archivo, así que se inyecta como columna literal (el contrato del
slot `(empresaid, direction, pair_rank)` lo requiere).

E4 se **monta** desde `16-e4-data-baselines`: este notebook no ejecuta el
preprocessing de E4 ni escribe `headways_E4.parquet`.

In [ ]:

headways = {}
for label, empresa_id in CORRIDORS:
    path = _resolve_input(f"headways_E{empresa_id}.parquet")
    frame = pl.read_parquet(path).with_columns(
        pl.lit(empresa_id, dtype=pl.Int64).alias("empresaid")
    )
    headways[label] = frame
    non_null = frame.filter(pl.col("delta_t_min").is_not_null()).height
    print(f"{label}: {frame.height:,} rows, {frame.width} cols, "
          f"non-null delta_t_min={non_null:,}  ({path})")

## Loop de export — 3 corredores × 4 horizontes

`export_paired_xgb` llama a `run_corridor` (split → winsorize train-only p99 →
B0-B4 → B5_XGB con random search sobre validación) y reencuadra las filas de
test al schema emparejado. Nada del pipeline se reimplementa aquí.

Son 12 ajustes en un solo kernel CPU (≈ NB10 + NB16). El CSV combinado se
reescribe al terminar cada corredor, para que una sesión interrumpida deje en
disco los corredores ya completados.

In [ ]:

paired_frames = []
provenance_rows = []

for label, empresa_id in CORRIDORS:
    for h in HORIZONS:
        started = time.time()
        paired, run = export_paired_xgb(
            headways[label], label, horizon=h, atypical_dates=atypical_dates
        )
        paired_frames.append(paired)
        provenance_rows.append(search_provenance_row(run, label, horizon=h))
        fit = run.fit_result
        print(f"{label} h={h}: {paired.height:,} paired test samples  "
              f"best_val_rmse={fit.best_val_rmse:.5f}  "
              f"({fit.n_configs_evaluated} configs, {time.time() - started:.1f}s)")

    # Flush after each corridor so an interrupted session keeps finished work.
    pl.concat(paired_frames).write_csv(PAIRED_OUT)
    print(f"  [flush] {label} done → {PAIRED_OUT}")

paired_export = pl.concat(paired_frames)
provenance = pl.DataFrame(provenance_rows)
print(f"\nTotal paired samples: {paired_export.height:,} "
      f"over {len(CORRIDORS)} corridors x {len(HORIZONS)} horizons")

## Verificación — la clave exportada ES única

El defecto que este notebook corrige es exactamente una clave no única: el export
histórico (`xgb_residuals_multih.csv`) usa `t` como si fuera clave, pero hay ~4.2
filas por `(t, direction)`. Aquí se afirma en runtime que
`(corridor, direction, horizon, t, pair_rank)` no tiene duplicados, y se
contrasta contra el conteo de la clave sin `pair_rank` para dejar el factor de
colapso en el log.

In [ ]:

n_rows = paired_export.height
n_full_key = paired_export.select(XGB_PAIRED_KEY).n_unique()
if n_full_key != n_rows:
    raise ValueError(
        f"paired export key is NOT unique: {n_rows:,} rows but {n_full_key:,} "
        f"distinct {XGB_PAIRED_KEY} tuples"
    )

key_without_pair_rank = [c for c in XGB_PAIRED_KEY if c != "pair_rank"]
n_collapsed = paired_export.select(key_without_pair_rank).n_unique()
print(f"Rows: {n_rows:,}")
print(f"Unique {XGB_PAIRED_KEY}: {n_full_key:,}  (must equal rows)")
print(f"Unique {key_without_pair_rank}: {n_collapsed:,} "
      f"→ {n_rows / max(n_collapsed, 1):.2f} rows per (t, direction) — this is why "
      f"pair_rank is required in the key")
print(paired_export.head(10))
print(paired_export.group_by(["corridor", "horizon"]).len().sort(["corridor", "horizon"]))

## Escribir CSVs (sólo CSV — este notebook NO escribe parquet)

1. `xgb_paired_persample_test.csv` — export emparejado por muestra
   (`corridor, empresaid, direction, horizon, t, pair_rank, y_true, y_pred_xgb,
   y_pred_persist`). Un único archivo combinado: `corridor` y `horizon` son parte
   de la clave, así que van como columnas y no como nombres de archivo.
2. `xgb_paired_search_provenance.csv` — configuración ganadora por
   (corredor, horizonte). Sirve para diferenciar este re-ajuste contra
   `xgb_search_config_multih.csv` / `xgb_search_config_E4_multih.csv`: si no
   coinciden, el export no es comparable y no debe usarse.

Ninguno de los dos nombres contiene `_results_`, `_residuals_h` ni `_multiseed_`,
para que ningún glob de la capa de análisis los descubra por accidente.

In [ ]:

paired_export.write_csv(PAIRED_OUT)
print(f"Paired export written to: {PAIRED_OUT}")
print(f"Rows: {paired_export.height:,}  Columns: {paired_export.columns}")

provenance.write_csv(PROVENANCE_OUT)
print(f"Search provenance written to: {PROVENANCE_OUT}")
print(provenance)